# Notebook 13: Differential Privacy Fundamentals

## Why This Matters

Every ML model trained on sensitive data carries an implicit question: *what guarantees can we make about what the model reveals about the individuals in its training set?* Membership inference, model inversion, and attribute inference attacks (notebooks 5-7) showed that the answer, without explicit protections, is "quite a lot."

Differential privacy (DP) is the first framework that provides a *mathematical answer* to this question. Instead of relying on intuition about what's safe, DP gives a formal bound: "the probability of any outcome changes by at most a factor of e^ε when any single individual's data is included or excluded." This epsilon (ε) is a hard, quantifiable guarantee — not a qualitative claim.

DP has become the privacy standard for census data (US Census Bureau switched to DP in 2020), mobile keyboard predictions (Apple's iOS keyboard, Google's Gboard), and increasingly for ML models. Understanding the fundamentals here is prerequisite to notebook 14 (DP-SGD) and the federated learning series.

## Background

### The Cynthia Dwork Origin Story

Cynthia Dwork introduced differential privacy in her 2006 paper *"Calibrating Noise to Sensitivity in Private Data Analysis."* The key motivation was a failure of prior approaches: *k*-anonymity and *l*-diversity (then the gold standard) had been repeatedly broken — you could re-identify individuals even in "anonymized" datasets by combining multiple quasi-identifiers.

Dwork's insight: instead of asking "can the adversary re-identify individual X?" (impossible to answer in general), ask "does including X's data in the dataset *change* what the adversary can learn?" If it doesn't, then X's privacy is preserved regardless of what auxiliary information the adversary has. This adversary-agnostic framing is the key innovation.

### The Formal Definition

A randomized mechanism M is (ε, δ)-differentially private if for all adjacent datasets D and D' (differing in exactly one individual), and all possible outputs S:

```
Pr[M(D) ∈ S] ≤ e^ε · Pr[M(D') ∈ S] + δ
```

- **ε (epsilon)**: the *privacy budget* — smaller is more private. ε=0 means the mechanism reveals nothing about any individual. ε=1 is a common target; ε>10 offers weak guarantees.
- **δ (delta)**: a small failure probability — the bound holds except with probability δ. Pure DP has δ=0. In practice δ=1/n² for dataset size n.
- **Adjacent datasets**: D and D' that differ in exactly one row. This models the "worst case" — one person's data.

### The Laplace Mechanism

For numeric queries with bounded *sensitivity* Δf (the maximum change in f(D) when one person is added/removed), the Laplace mechanism achieves ε-DP:

```
M(D) = f(D) + Lap(Δf / ε)
```

The noise scale is *proportional to sensitivity and inversely proportional to budget*. High sensitivity → more noise. Tight budget (small ε) → more noise. This is the fundamental privacy-utility tradeoff.

### The Gaussian Mechanism

The Gaussian mechanism achieves (ε, δ)-DP (approximate DP):

```
M(D) = f(D) + N(0, σ²)
σ = Δ₂f · √(2 ln(1.25/δ)) / ε
```

where Δ₂f is the L2 sensitivity. The Gaussian mechanism is key for DP-SGD because it adds isotropic Gaussian noise to gradient vectors, which have natural L2 structure.

### Composition Theorems

When you run multiple DP mechanisms sequentially (as in training for many epochs), the privacy budgets compose. The *basic composition theorem* says that k mechanisms with budget ε each compose to kε-DP. This is a worst case — the *advanced composition theorem* and *Rényi DP* provide tighter bounds. In practice, ML training uses moments accountant or RDP accounting to compute the actual privacy spent over thousands of gradient steps.

### Local vs. Central DP

- **Central DP**: A trusted curator collects raw data, runs a DP mechanism, and publishes noisy results. The privacy guarantee is against external observers of the published output. This is what the census uses.
- **Local DP**: Each individual adds noise to their own data *before* sending it to the curator. The curator sees only noisy data. This requires more noise (worse utility) but provides stronger protection — even the curator can't learn individual values. This is what Apple uses for iOS keyboard prediction.

## Structure of This Notebook

1. Formal DP definition and adjacent datasets
2. Sensitivity analysis — global vs. local sensitivity
3. Laplace mechanism with visualization
4. Gaussian mechanism with (ε, δ) bounds
5. Composition: basic, advanced, Rényi DP
6. Randomized response — local DP on binary attributes
7. Privacy amplification by subsampling

## What You'll Learn

- The formal definition of (ε, δ)-DP and how to read it
- Why sensitivity determines noise scale
- How Laplace vs. Gaussian mechanisms trade off ε and δ
- How composition degrades privacy over multiple queries
- Randomized response as a local DP primitive
- Why subsampling amplifies privacy guarantees

In [ ]:
%pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports ready.')

## 1. What DP Actually Guarantees — A Visual Intuition

The DP guarantee is about the *ratio* of probabilities. If the mechanism's output distribution barely changes when one person is added or removed, an adversary can't use the output to determine whether that person was in the dataset.

In [ ]:
def demonstrate_dp_bound(epsilon=1.0, sensitivity=1.0, n_samples=100_000):
    """
    Visualize that the DP guarantee holds:
    For all outputs s: Pr[M(D)=s] / Pr[M(D')=s] ≤ e^ε
    
    We use a simple count query with Laplace mechanism.
    D = dataset where person is IN (count=50)
    D' = adjacent dataset where person is OUT (count=49)
    """
    true_count_D = 50
    true_count_Dprime = 49   # adjacent dataset: one person removed
    scale = sensitivity / epsilon  # Laplace noise scale

    # Sample from both mechanisms
    samples_D = true_count_D + np.random.laplace(0, scale, n_samples)
    samples_Dprime = true_count_Dprime + np.random.laplace(0, scale, n_samples)

    # Estimate probability density at each output value
    x_range = np.linspace(40, 60, 500)
    kde_D = stats.gaussian_kde(samples_D)
    kde_Dprime = stats.gaussian_kde(samples_Dprime)

    p_D = kde_D(x_range)
    p_Dprime = kde_Dprime(x_range)

    # Compute ratio — should be bounded by e^ε everywhere
    ratio = p_D / (p_Dprime + 1e-12)
    ratio_clipped = np.clip(ratio, 0, np.exp(epsilon) * 2)  # clip for display

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Plot distributions
    axes[0].plot(x_range, p_D, 'b-', label=f'M(D) — person IN, count={true_count_D}')
    axes[0].plot(x_range, p_Dprime, 'r--', label=f'M(D\') — person OUT, count={true_count_Dprime}')
    axes[0].fill_between(x_range, p_D, p_Dprime, alpha=0.2)
    axes[0].set_xlabel('Mechanism Output (Noisy Count)')
    axes[0].set_ylabel('Probability Density')
    axes[0].set_title(f'Laplace Mechanism (ε={epsilon}, Δ={sensitivity})')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Plot ratio
    axes[1].plot(x_range, ratio_clipped, 'g-', label='Pr[M(D)] / Pr[M(D\')]')
    axes[1].axhline(np.exp(epsilon), color='red', linestyle='--',
                    label=f'DP bound: e^ε = {np.exp(epsilon):.2f}')
    axes[1].axhline(1/np.exp(epsilon), color='orange', linestyle='--',
                    label=f'Lower bound: 1/e^ε = {1/np.exp(epsilon):.2f}')
    axes[1].set_xlabel('Output Value')
    axes[1].set_ylabel('Probability Ratio')
    axes[1].set_title('Ratio of Probabilities — Bounded by e^ε')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, np.exp(epsilon) * 1.5)

    plt.tight_layout()
    plt.show()

    max_ratio = ratio[(p_D > 0.001) & (p_Dprime > 0.001)].max()
    print(f'Max observed ratio: {max_ratio:.3f} | DP bound e^ε: {np.exp(epsilon):.3f}')
    print(f'Bound satisfied: {max_ratio <= np.exp(epsilon) * 1.05}')

demonstrate_dp_bound(epsilon=1.0, sensitivity=1.0)

## 2. Sensitivity — The Key to Calibrating Noise

In [ ]:
def global_sensitivity_demo():
    """
    Demonstrate L1 and L2 sensitivity for common queries.
    Sensitivity = max over adjacent D, D' of |f(D) - f(D')|.
    """
    n = 1000
    ages = np.random.randint(18, 80, n)  # dataset: n people's ages

    print('Sensitivity analysis for common queries:')
    print('=' * 55)

    # Count query: f(D) = |D|
    # Adding/removing one person changes count by 1 → sensitivity = 1
    print(f'COUNT query: sensitivity = 1 (always)')

    # Sum query: f(D) = sum of ages
    # Adding/removing one person can change sum by up to max_age - min_age
    age_min, age_max = 18, 80
    sum_sensitivity = age_max - age_min
    print(f'SUM(age) query: sensitivity = max_age - min_age = {sum_sensitivity}')

    # Mean query: f(D) = mean(D)
    # Adding/removing one person with value v: |mean(D ∪ {v}) - mean(D)|
    # In worst case: |v - mean(D)| / (n+1) ≈ range / n
    mean_sensitivity = (age_max - age_min) / n
    print(f'MEAN(age) query: sensitivity ≈ range/n = {mean_sensitivity:.4f}')
    print(f'  (Mean is much less sensitive than sum — averaging dilutes individual impact)')

    # Histogram query: f(D) = vector of counts per bin
    # L1 sensitivity: one person moves from one bin to another → L1 change = 2
    # (count in one bin -1, another +1)
    print(f'HISTOGRAM query: L1 sensitivity = 2 (one person leaves one bin, enters another)')

    print()
    print('Noise scales for ε=1.0:')
    for name, sens in [('COUNT', 1), ('SUM(age)', sum_sensitivity),
                        ('MEAN(age)', mean_sensitivity), ('HISTOGRAM', 2)]:
        noise_scale = sens / 1.0  # ε=1.0
        print(f'  {name:<15}: Lap(0, {noise_scale:.4f})')

global_sensitivity_demo()

In [ ]:
# Sensitivity of gradient clipping in DP-SGD context (preview of notebook 14)
def demonstrate_gradient_sensitivity(clip_norm=1.0, n_params=1000):
    """
    In DP-SGD, gradients are clipped to L2 norm C before Gaussian noise is added.
    This bounds the L2 sensitivity of the gradient to C.
    """
    # Simulate per-sample gradients
    gradients = np.random.randn(10, n_params)  # 10 samples
    norms = np.linalg.norm(gradients, axis=1)

    # Clip to L2 norm C
    clipped = gradients / np.maximum(norms[:, None] / clip_norm, 1)
    clipped_norms = np.linalg.norm(clipped, axis=1)

    print(f'Gradient clipping (C={clip_norm}):')
    print(f'  Before clip: norms = {norms.round(3)}')
    print(f'  After clip:  norms = {clipped_norms.round(3)}')
    print(f'  L2 sensitivity of clipped sum = {clip_norm} (guaranteed by clip)')
    print(f'  Gaussian noise sigma = C * sqrt(2*ln(1.25/δ)) / ε')
    epsilon, delta = 1.0, 1e-5
    sigma = clip_norm * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    print(f'  For ε={epsilon}, δ={delta}: sigma = {sigma:.3f}')

demonstrate_gradient_sensitivity()

## 3. Laplace Mechanism

In [ ]:
class LaplaceMechanism:
    """Pure DP mechanism for real-valued queries via Laplace noise."""

    def __init__(self, sensitivity: float, epsilon: float):
        self.sensitivity = sensitivity
        self.epsilon = epsilon
        self.scale = sensitivity / epsilon  # noise scale b

    def apply(self, true_value):
        """Return f(D) + Lap(0, Δf/ε)."""
        noise = np.random.laplace(0, self.scale)
        return true_value + noise

    def apply_vector(self, true_vector):
        """Apply independent Laplace noise to each component (L1 sensitivity)."""
        noise = np.random.laplace(0, self.scale, size=true_vector.shape)
        return true_vector + noise


# Demonstrate accuracy vs. epsilon tradeoff
n_people = 1000
ages = np.random.randint(18, 80, n_people)
true_mean_age = ages.mean()

# Mean query: sensitivity = range / n
age_sensitivity = (80 - 18) / n_people
epsilons = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

print(f'True mean age: {true_mean_age:.2f}')
print()
print(f'{"ε":>6} {"Noise scale":>12} {"Noisy mean (avg 100 runs)":>26} {"MAE":>8}')
print('-' * 60)

results = []
for eps in epsilons:
    mech = LaplaceMechanism(age_sensitivity, eps)
    estimates = [mech.apply(true_mean_age) for _ in range(100)]
    mean_est = np.mean(estimates)
    mae = np.mean(np.abs(np.array(estimates) - true_mean_age))
    results.append((eps, mech.scale, mean_est, mae))
    print(f'{eps:>6.2f} {mech.scale:>12.4f} {mean_est:>26.4f} {mae:>8.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epsilons, [r[3] for r in results], 'b-o')
ax.set_xlabel('Privacy Budget ε (larger = less private)')
ax.set_ylabel('Mean Absolute Error of Estimate')
ax.set_title('Laplace Mechanism: Privacy-Utility Tradeoff (Mean Age Query)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Gaussian Mechanism — Approximate DP

In [ ]:
class GaussianMechanism:
    """
    (ε, δ)-DP mechanism for L2-sensitive queries.
    Preferred for high-dimensional outputs (e.g., gradient vectors).
    """

    def __init__(self, l2_sensitivity: float, epsilon: float, delta: float):
        self.l2_sensitivity = l2_sensitivity
        self.epsilon = epsilon
        self.delta = delta
        # Standard sigma formula from Dwork & Roth
        self.sigma = l2_sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon

    def apply(self, true_value):
        noise = np.random.normal(0, self.sigma)
        return true_value + noise

    def apply_vector(self, true_vector):
        noise = np.random.normal(0, self.sigma, size=true_vector.shape)
        return true_vector + noise


# Compare Laplace vs. Gaussian at same ε
eps = 1.0
delta = 1e-5
sensitivity = 1.0

lap_mech = LaplaceMechanism(sensitivity, eps)
gauss_mech = GaussianMechanism(sensitivity, eps, delta)

print(f'Comparing mechanisms at ε={eps}, δ={delta}:')
print(f'  Laplace noise scale b = {lap_mech.scale:.4f}')
print(f'  Gaussian noise σ = {gauss_mech.sigma:.4f}')
print()
print(f'  Laplace std dev = {lap_mech.scale * np.sqrt(2):.4f}  (std = b*sqrt(2))')
print(f'  Gaussian std dev = {gauss_mech.sigma:.4f}')
print()
print(f'The Gaussian mechanism adds more noise per query, but achieves (ε,δ)-DP')
print(f'rather than pure ε-DP. This is worthwhile for high-dimensional vectors:')
print(f'  - For a 1M-parameter gradient vector, adding Laplace noise to each coordinate')
print(f'    requires L1 sensitivity analysis (hard), while Gaussian uses L2 (natural).')

# Visualize the two noise distributions
x = np.linspace(-5, 5, 1000)
lap_pdf = stats.laplace.pdf(x, scale=lap_mech.scale)
gauss_pdf = stats.norm.pdf(x, scale=gauss_mech.sigma)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, lap_pdf, 'b-', label=f'Laplace(0, {lap_mech.scale:.2f}) — pure ε-DP')
ax.plot(x, gauss_pdf, 'r-', label=f'Gaussian(0, {gauss_mech.sigma:.2f}²) — (ε,δ)-DP')
ax.set_xlabel('Noise Value')
ax.set_ylabel('PDF')
ax.set_title(f'Laplace vs. Gaussian Noise (ε={eps}, δ={delta})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Composition Theorems

Training an ML model runs the mechanism (gradient computation) *thousands of times*. Each run spends privacy budget. Composition theorems track how budget accumulates.

In [ ]:
def basic_composition(epsilon_per_step, n_steps):
    """Basic: k steps of ε-DP → kε-DP."""
    return epsilon_per_step * n_steps


def advanced_composition(epsilon_per_step, delta_per_step, n_steps, delta_total):
    """
    Advanced composition (Dwork et al. 2010):
    k steps of (ε, δ)-DP compose to (ε', k*δ + δ_total)-DP where:
    ε' = sqrt(2k * ln(1/δ_total)) * ε + k*ε*(e^ε - 1)
    Significantly better than basic for large k.
    """
    e = epsilon_per_step
    k = n_steps
    epsilon_adv = np.sqrt(2 * k * np.log(1 / delta_total)) * e + k * e * (np.exp(e) - 1)
    delta_adv = k * delta_per_step + delta_total
    return epsilon_adv, delta_adv


def rdp_composition_gaussian(sigma, q, n_steps, target_delta):
    """
    Simplified Rényi DP accountant for Gaussian mechanism.
    q = sampling ratio (batch_size / dataset_size)
    This approximates the moments accountant used in DP-SGD.

    RDP guarantees: (α, ε_α)-RDP composes as sum of ε_α over steps,
    then converts to (ε, δ)-DP.
    """
    # Simplified: use the single-step RDP for Gaussian
    # ε_α(Gaussian) ≈ α / (2σ²) for the subsampled mechanism
    # This is a rough approximation for illustration
    alpha_values = np.arange(2, 50)
    best_epsilon = np.inf

    for alpha in alpha_values:
        # Per-step RDP for subsampled Gaussian (simplified)
        rdp_per_step = q**2 * alpha / (sigma**2)
        total_rdp = rdp_per_step * n_steps
        # Convert RDP to (ε, δ)-DP
        epsilon = total_rdp + np.log(1 / target_delta) / (alpha - 1)
        best_epsilon = min(best_epsilon, epsilon)

    return best_epsilon


# Example: DP-SGD training for 100 epochs, 500 steps/epoch
n_steps = 100 * 500
epsilon_per_step = 0.01
delta_per_step = 1e-8
target_delta = 1e-5

eps_basic = basic_composition(epsilon_per_step, n_steps)
eps_adv, delta_adv = advanced_composition(epsilon_per_step, delta_per_step, n_steps, target_delta)
eps_rdp = rdp_composition_gaussian(sigma=1.0, q=0.01, n_steps=n_steps, target_delta=target_delta)

print(f'Training: {n_steps} steps, ε per step = {epsilon_per_step}')
print()
print(f'Basic composition:     ε = {eps_basic:.2f} (worst case — multiply)')
print(f'Advanced composition:  ε = {eps_adv:.4f}, δ = {delta_adv:.2e}')
print(f'RDP accountant:        ε ≈ {eps_rdp:.4f} (tightest bound)')
print()
print(f'Improvement over basic: {eps_basic/eps_rdp:.1f}x tighter with RDP')

# Show how epsilon grows with number of training steps
step_counts = np.logspace(1, 5, 50).astype(int)
eps_basic_list = [basic_composition(epsilon_per_step, k) for k in step_counts]
eps_adv_list = [advanced_composition(epsilon_per_step, delta_per_step, k, target_delta)[0]
                for k in step_counts]
eps_rdp_list = [rdp_composition_gaussian(1.0, 0.01, k, target_delta) for k in step_counts]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(step_counts, eps_basic_list, 'r-', label='Basic composition (kε)')
ax.plot(step_counts, eps_adv_list, 'b-', label='Advanced composition')
ax.plot(step_counts, eps_rdp_list, 'g-', label='RDP accountant (Gaussian)')
ax.axvline(n_steps, color='gray', linestyle='--', alpha=0.5, label=f'{n_steps} steps (100 epochs)')
ax.set_xlabel('Number of Training Steps')
ax.set_ylabel('Total Privacy Budget ε')
ax.set_title('Privacy Budget Accumulation Under Composition')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Randomized Response — Local Differential Privacy

In [ ]:
def randomized_response(true_answer: bool, epsilon: float) -> bool:
    """
    Local DP mechanism for binary attributes (Warner 1965, formalized by Dwork).

    Protocol:
    - Flip a biased coin with p = e^ε / (e^ε + 1)
    - If heads: report truth
    - If tails: report randomly (50/50)

    Achieves ε-LDP. The server can estimate the true population proportion
    without knowing any individual's true answer.
    """
    p_truth = np.exp(epsilon) / (np.exp(epsilon) + 1)
    if np.random.random() < p_truth:
        return true_answer
    else:
        return bool(np.random.randint(2))


def estimate_population_proportion(responses: list, epsilon: float) -> float:
    """
    Debias randomized response to estimate true proportion.

    If true proportion p, observed proportion p* satisfies:
    p* = p * (e^ε/(e^ε+1)) + (1-p) * (1/(e^ε+1))
    Solving for p:
    p = (p* * (e^ε + 1) - 1) / (e^ε - 1)
    """
    p_star = np.mean(responses)
    e_eps = np.exp(epsilon)
    p_true_hat = (p_star * (e_eps + 1) - 1) / (e_eps - 1)
    return float(np.clip(p_true_hat, 0, 1))


# Simulate: 10k people, 30% have a sensitive attribute (e.g., "has used illegal drugs")
n = 10_000
true_prop = 0.30
true_answers = np.random.random(n) < true_prop

print(f'True proportion: {true_prop:.3f}')
print(f'True count (known only to us): {true_answers.sum()}')
print()
print(f'{"ε":>6} {"Responses Y":>12} {"Raw estimate":>13} {"Debiased estimate":>18} {"Error":>8}')
print('-' * 65)

epsilon_values = [0.1, 0.5, 1.0, 2.0, 5.0]
for eps in epsilon_values:
    responses = [randomized_response(bool(a), eps) for a in true_answers]
    raw_est = np.mean(responses)
    debiased = estimate_population_proportion(responses, eps)
    error = abs(debiased - true_prop)
    print(f'{eps:>6.1f} {sum(responses):>12} {raw_est:>13.4f} {debiased:>18.4f} {error:>8.4f}')

print()
print('The debiased estimate is unbiased regardless of ε.')
print('Lower ε → more noise per response → higher variance in estimate.')
print('Key property: the server learns population statistics without seeing individual answers.')

## 7. Privacy Amplification by Subsampling

When you run a mechanism on a *random subsample* of the dataset rather than the full dataset, the effective privacy guarantee improves. This is the key insight behind mini-batch training in DP-SGD.

In [ ]:
def amplified_epsilon(epsilon_base, sampling_ratio):
    """
    Subsampling amplification theorem (Balle et al. 2018 tight bound for Poisson):
    If mechanism M is ε-DP, then the subsampled version with ratio q is:
    ε_amplified = ln(1 + q(e^ε - 1))

    For small q and ε: ε_amplified ≈ q * ε  (approximately linear in sampling ratio)
    """
    return np.log(1 + sampling_ratio * (np.exp(epsilon_base) - 1))


# Show amplification for different batch sizes
n_total = 60_000  # MNIST training set
batch_sizes = [64, 128, 256, 512, 1024, 4096]
epsilon_base = 1.0  # per-step ε without subsampling

print(f'Privacy amplification via subsampling (dataset size n={n_total})')
print(f'Base per-step ε = {epsilon_base}')
print()
print(f'{"Batch size":>12} {"q (ratio)":>10} {"Amplified ε":>13} {"Amplification":>15}')
print('-' * 55)

for bs in batch_sizes:
    q = bs / n_total
    eps_amp = amplified_epsilon(epsilon_base, q)
    amp_factor = epsilon_base / eps_amp
    print(f'{bs:>12} {q:>10.5f} {eps_amp:>13.6f} {amp_factor:>15.1f}x')

# Visualize: amplification across sampling ratios
q_values = np.linspace(0.001, 1.0, 500)
eps_amplified = [amplified_epsilon(epsilon_base, q) for q in q_values]
eps_naive = [epsilon_base * q for q in q_values]  # naive linear approximation

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(q_values, eps_amplified, 'b-', label='Amplified ε (exact)')
ax.plot(q_values, eps_naive, 'r--', label='Naive: q·ε (approximation)')
ax.axhline(epsilon_base, color='gray', linestyle=':', label=f'Base ε (no subsampling)')
ax.set_xlabel('Sampling Ratio q = batch_size / n')
ax.set_ylabel('Effective Privacy Budget ε')
ax.set_title('Privacy Amplification by Subsampling')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('Key insight for DP-SGD:')
print('  - Mini-batch training naturally subsamples the dataset at each step')
print('  - Each gradient update only touches a fraction q of training data')
print('  - The effective per-step ε is amplified by ~q (for small q)')
print('  - Smaller batches → stronger privacy amplification → lower total ε')

## 8. Putting It Together — Privacy Budget Calculator

In [ ]:
def privacy_budget_calculator(
    n_samples: int,
    batch_size: int,
    n_epochs: int,
    noise_multiplier: float,
    target_delta: float = 1e-5,
):
    """
    Estimate total privacy cost (ε) for a DP-SGD training run.
    Uses simplified RDP accountant.

    noise_multiplier = σ / C (sigma divided by clip norm).
    Real implementations use Opacus or tensorflow-privacy for exact accounting.
    """
    q = batch_size / n_samples  # sampling ratio
    steps_per_epoch = n_samples // batch_size
    total_steps = steps_per_epoch * n_epochs

    # Use RDP accountant (simplified)
    epsilon = rdp_composition_gaussian(
        sigma=noise_multiplier, q=q, n_steps=total_steps, target_delta=target_delta
    )

    return {
        'n_samples': n_samples,
        'batch_size': batch_size,
        'sampling_ratio': q,
        'total_steps': total_steps,
        'noise_multiplier': noise_multiplier,
        'epsilon': epsilon,
        'delta': target_delta,
    }


# MNIST-scale training under various privacy budgets
configs = [
    {'noise_multiplier': 0.5, 'n_epochs': 20},
    {'noise_multiplier': 1.0, 'n_epochs': 20},
    {'noise_multiplier': 1.5, 'n_epochs': 20},
    {'noise_multiplier': 1.0, 'n_epochs': 10},
    {'noise_multiplier': 1.0, 'n_epochs': 50},
]

print('Privacy budget for MNIST-scale DP-SGD training:')
print(f'n=60k, batch=256, δ=1e-5')
print()
print(f'{"Noise mult":>10} {"Epochs":>8} {"Steps":>8} {"ε":>8} {"Privacy level":>15}')
print('-' * 58)

for cfg in configs:
    result = privacy_budget_calculator(
        n_samples=60_000,
        batch_size=256,
        n_epochs=cfg['n_epochs'],
        noise_multiplier=cfg['noise_multiplier'],
    )
    eps = result['epsilon']
    level = 'Strong' if eps < 1 else 'Moderate' if eps < 5 else 'Weak'
    print(f"{cfg['noise_multiplier']:>10.1f} {cfg['n_epochs']:>8} {result['total_steps']:>8} "
          f"{eps:>8.3f} {level:>15}")

## Summary

### What We Built

| Concept | Implementation | Key Formula |
|---------|---------------|-------------|
| DP bound visualization | KDE ratio plot | `Pr[M(D)] / Pr[M(D')] ≤ e^ε` |
| Sensitivity analysis | Query-by-query analysis | `Δf = max |f(D) - f(D')|` |
| `LaplaceMechanism` | Additive Lap(0, Δf/ε) noise | Pure ε-DP |
| `GaussianMechanism` | Additive N(0, σ²) noise | (ε,δ)-DP |
| Composition | Basic / Advanced / RDP | RDP is 10-100x tighter |
| `randomized_response` | Local DP for binary attributes | ε-LDP without trusted curator |
| `amplified_epsilon` | Subsampling privacy amplification | `ln(1 + q(e^ε - 1))` |
| `privacy_budget_calculator` | Full training cost estimate | Simplified RDP accountant |

### Key Takeaways

- **ε is a hard mathematical bound**, not a qualitative claim. ε=1 means the likelihood ratio of any outcome between adjacent datasets is at most e≈2.72.
- **Sensitivity determines noise scale.** High-sensitivity queries (sums, argmax) require more noise. Low-sensitivity queries (means, histograms) can be answered more accurately.
- **Composition degrades privacy.** Running k mechanisms naively costs kε. The RDP accountant (used in Opacus/tensorflow-privacy) provides 10-100x tighter bounds.
- **Subsampling amplifies privacy.** Training with mini-batches gives a free privacy dividend — each step only touches fraction q of the data, reducing effective ε by ~q.
- **Local DP needs no trusted server** but costs more noise. Central DP achieves better accuracy at the cost of trusting the data curator.

### The Fundamental Tradeoff

There is no free lunch: stronger privacy (smaller ε) requires more noise, which degrades utility. The goal of DP-SGD (notebook 14) is to navigate this tradeoff during gradient descent — adding exactly enough noise to the gradients to achieve a target ε while preserving as much model accuracy as possible.

Next: Notebook 14 implements DP-SGD — differentially private stochastic gradient descent — using per-sample gradient clipping and Gaussian noise addition, with full privacy accounting.